# Health Trainer — Colab Training Runner

Thin runner: the real logic lives in `ml/src/` (source of truth). This notebook mounts Drive, clones the `model-training` branch, installs dependencies, and writes experiment artifacts into `health_training/runs/`.

First real-data target: KaggleHub `thashmiladewmini/squat-exercise-pose-dataset` -> `train_squat_form_classifier.py`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
DATA_DIR   = f'{DRIVE_ROOT}/data'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
EXPORT_DIR = f'{DRIVE_ROOT}/exports/latest'
print(DRIVE_ROOT)


In [ ]:
# Source of truth: GitHub model-training branch.
!rm -rf /content/health_trainer
!git clone --branch model-training --single-branch https://github.com/kimgt0128/health-trainer.git /content/health_trainer
%cd /content/health_trainer
!pip install -q -r ml/requirements-colab.txt


In [ ]:
import os
os.environ['PYTHONPATH'] = '/content/health_trainer/ml/src'

# Track A: real-data squat form baseline.
# Uses KaggleHub inside train_squat_form_classifier.py:
#   thashmiladewmini/squat-exercise-pose-dataset
#   squat_dataset/squat_features_augmented.csv
!python ml/src/train_squat_form_classifier.py \
    --run-dir "$RUNS_DIR/squat_form_classifier_rf_v1" \
    --test-size 0.2 \
    --random-state 42


In [ ]:
# Inspect artifacts written to Drive.
!find "$RUNS_DIR/squat_form_classifier_rf_v1" -maxdepth 1 -type f -print
!cat "$RUNS_DIR/squat_form_classifier_rf_v1/metrics_summary.json"


In [ ]:
# Track B: raw video -> MediaPipe landmarks -> sequence models.
# Use only after Drive data/raw/ has real videos.
# !python ml/src/extract_landmarks.py --input-dir "$DATA_DIR/raw/public" \
#     --output-dir "$DATA_DIR/landmarks/public" --model pose_landmarker_lite.task
# !python ml/src/build_features.py --landmarks-dir "$DATA_DIR/landmarks/public" \
#     --config ml/configs/exercise_classifier.yaml --out "$DATA_DIR/splits/exercise_train.npz"
# !python ml/src/train_exercise_classifier.py --features "$DATA_DIR/splits/exercise_train.npz" \
#     --run-dir "$RUNS_DIR/exercise_classifier_rf_v1" --model baseline
